# Лаборатория 11 — Трекер сделок

Каркас для итогового модуля: соберите **бумажный портфель первой недели**, прогоните `summarize` по
каждой позиции и получите один **компактный отчёт по портфелю** — DataFrame с метриками по каждой
позиции плюс **суммарные греки книги**. Разверните это в свой рабочий трекер на 30-дневную программу.

Книга первой недели (на DEMO, спот 100, IV ~25%) намеренно смешивает требуемое разнообразие:
**доходная** конструкция, **направленная** сделка и **временной спред**. Работает офлайн, сверху
вниз.

In [ ]:
import numpy as np
import pandas as pd
from optionslab import strategies, analyzer, greeks
SPOT, VOL = 100.0, 0.26

## 1. Собираем портфель первой недели

Одна доходная / нейтральная позиция (айрон кондор), одна направленная (бычий колл-дебетовый спред),
один временной спред (календарь). Размер каждой был бы отмерен под 1-2% максимального убытка, и
каждая прошла бы чек-лист входа.

In [ ]:
book = {
    "income": strategies.iron_condor((87.5,0.37),(92.5,1.01),(107.5,1.19),(112.5,0.43), expiry=45/365),
    "directional": strategies.bull_call_spread((100,3.91),(110,0.73), expiry=45/365),
    "time_spread": strategies.calendar_spread("call", 100, front_expiry=21/365, front_premium=2.66,
                                              back_expiry=45/365, back_premium=3.91),
}
for tag, pos in book.items():
    print(f"{tag:12} {pos.label}")

## 2. Сводка по каждой позиции

`analyzer.summarize` возвращает чистую премию, точки безубыточности, максимальные прибыль/убыток,
POP, ожидаемое движение, греки и DTE. Соберём эти поля в одну аккуратную строку на позицию.

In [ ]:
def row(tag, pos, spot, vol):
    s = analyzer.summarize(pos, spot, vol)
    g = s["greeks"]
    return {
        "tag": tag, "label": s["label"][:26],
        "net$": round(s["net_premium"], 0), "maxP$": round(s["max_profit"], 0),
        "maxL$": round(s["max_loss"], 0), "POP": round(s["probability_of_profit"], 2),
        "DTE": s["days_to_expiry"],
        "delta": round(g.delta, 1), "theta": round(g.theta, 1), "vega": round(g.vega, 1),
    }

In [ ]:
report = pd.DataFrame([row(t, p, SPOT, VOL) for t, p in book.items()])
report

## 3. Суммарные греки книги и итоги

Сложите долларовые греки по всей книге (они складываются) и просуммируйте колонки риска, чтобы
получить картину портфеля одним взглядом.

In [ ]:
total_g = greeks.Greeks(0, 0, 0, 0, 0)
for pos in book.values():
    total_g = total_g + greeks.position_greeks(pos, SPOT, VOL)
print("КНИГА delta/theta/vega:",
      round(total_g.delta, 1), round(total_g.theta, 1), round(total_g.vega, 1))
print("КНИГА: максимальный убыток, если все дойдут до худшего случая $:", round(report["maxL$"].sum(), 0))

Суммарные греки показывают реальную позу книги (здесь: чистая длинная дельта из направленного
спреда и смесь по тете/веге от кондора и календаря). Сверяйте `delta` и `vega` с лимитами портфеля из
вашего торгового плана, прежде чем добавлять что-то новое.

In [ ]:
LIMITS = {"delta": (-200, 200), "vega": (-50, 50)}
for k in ("delta", "vega"):
    v = getattr(total_g, k); lo, hi = LIMITS[k]
    print(f"чистая {k} {round(v,1):>7}  лимит [{lo}, {hi}]  ->", "ОК" if lo <= v <= hi else "ПРОБОЙ")

## 4. Компактный печатный отчёт

Один блок, который можно вставить в ежедневный журнал: риск по каждой позиции + итоги книги.

In [ ]:
print(report.to_string(index=False))
print("-" * 60)
print(f"КНИГА  net$={report['net$'].sum():.0f}  maxL$={report['maxL$'].sum():.0f}  "
      f"delta={total_g.delta:.1f}  theta={total_g.theta:.1f}  vega={total_g.vega:.1f}")

## Эксперименты

1. Добавьте четвёртую позицию (обеспеченный деньгами пут или диагональ) и перезапустите — смотрите,
   как обновятся греки книги и суммарный максимальный убыток. Всё ещё укладывается в ваши лимиты?
2. Добавьте колонку **days_forward**: для каждой позиции посчитайте
   `payoff.pnl_at(pos, SPOT, 10/365, VOL)`, чтобы увидеть P&L по модели через 10 дней без движения
   цены (тета за работой).
3. Замените направленную ногу на **медвежью** и убедитесь, что чистая дельта книги меняет знак.
4. Проследите ту же книгу на движении цены: задайте `SPOT = 96` и перезапустите — у какой позиции
   греки меняются сильнее всего и какая теперь протестирована?
5. Превратите `row()` в свой рабочий трекер: добавьте колонки под дату входа, цель по прибыли и флаг
   «статус относительно плана» и дописывайте строку каждый раз, когда открываете бумажную сделку в
   30-дневной программе.